In [ ]:
import pandas as pd
import cloudscraper
from recipe_scrapers import scrape_html
import requests
import time
import re
import os
import random
import threading
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
os.makedirs("recipe_images", exist_ok=True)
rec_df = pd.read_csv("cleaned_dishes.csv")

start_index = 99900
remaining_df = rec_df.iloc[start_index:]

thread_local = threading.local()

def get_scraper():
    if not hasattr(thread_local, "scraper"):
        thread_local.scraper = cloudscraper.create_scraper(
            browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True}
        )
    return thread_local.scraper

In [ ]:
def process_recipe(index, row):
    recipe_id = str(row["id"])
    raw_name = str(row["name"])
    file_path = f"recipe_images/{recipe_id}.jpg"
    
    if os.path.exists(file_path):
        return index, "skips (logo + seen)"
    
    slug = raw_name.lower()
    slug = re.sub(r'[^a-z0-9\s\-]', '', slug)  
    slug = re.sub(r'[\s\-]+', '-', slug).strip('-') 
    url = f"https://www.food.com/recipe/{slug}-{recipe_id}"
    
    scraper = get_scraper()
    
    try:
        with scraper.get(url, timeout=15) as response:
            if response.status_code == 200:
                recipe_parser = scrape_html(html=response.text, org_url=url)
                image_url = recipe_parser.image()
                
                if image_url:
                    if "default.jpg" in image_url.lower():
                        return index, "skips (logo + seen)"
                    else:
                        with requests.get(image_url, stream=True, timeout=10) as img_response:
                            if img_response.status_code == 200:
                                with open(file_path, 'wb') as f:
                                    for chunk in img_response.iter_content(1024):
                                        f.write(chunk)
                                        
                                bad_bytes = [113433] 
                                file_size = os.path.getsize(file_path)
                                
                                if file_size in bad_bytes:
                                    os.remove(file_path) 
                                    return index, "skips (logo + seen)"
                                else:
                                    time.sleep(random.uniform(0.4, 0.6))
                                    return index, "downloads"
                            else:
                                return index, "errors"
                else:
                    return index, "skips (logo + seen)"
                        
            elif response.status_code == 404:
                return index, "skips (404)"
                
            else:
                return index, f"blocked_{response.status_code}"
            
    except Exception as e:
        return index, f"error_{e}"

In [ ]:
stats = {"idx": 0, "downloads": 0, "skips (logo + seen)": 0, "skips (404)": 0, "errors": 0}
MAX_WORKERS = 8 

print(f"Unleashing {MAX_WORKERS} threads on {remaining_df.shape[0]} recipes...")

with tqdm(total=remaining_df.shape[0], desc="scraping images", unit="img") as pbar:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_recipe, index, row): index for index, row in remaining_df.iterrows()}
        
        for future in as_completed(futures):
            index, result = future.result()
            
            stats["idx"] = max(stats["idx"], index) 
            
            if result in stats:
                stats[result] += 1
            elif str(result).startswith("blocked"):
                stats["errors"] += 1
                tqdm.write(f"[{index}] Blocked. Status: {result.split('_')[1]}")
            elif str(result).startswith("error"):
                stats["errors"] += 1
                tqdm.write(f"[{index}] Error: {result.split('_')[1]}")
            
            pbar.update(1)
            
            if pbar.n % 20 == 0:
                pbar.set_postfix(stats)

pbar.set_postfix(stats)

Unleashing 8 threads on 130267 recipes...


scraping images:   4%|▍         | 5822/130267 [21:07<5:43:06,  6.04img/s, idx=105720, downloads=3026, skips (logo + seen)=2779, skips (404)=7, errors=8] 

[105725] Blocked. Status: 500


scraping images:   6%|▌         | 7368/130267 [26:34<8:06:50,  4.21img/s, idx=107263, downloads=3862, skips (logo + seen)=3482, skips (404)=7, errors=9] 

[107270] Blocked. Status: 500


...

scraping images:  98%|█████████▊| 127314/130267 [6:54:44<11:12,  4.39img/s, idx=227203, downloads=66002, skips (logo + seen)=60905, skips (404)=253, errors=140]  

[227214] Blocked. Status: 500


scraping images:  99%|█████████▉| 129256/130267 [7:00:53<02:25,  6.94img/s, idx=229142, downloads=67040, skips (logo + seen)=61803, skips (404)=255, errors=142]

[229161] Blocked. Status: 500


scraping images: 100%|██████████| 130267/130267 [7:04:16<00:00,  5.12img/s, idx=230160, downloads=67557, skips (logo + seen)=62304, skips (404)=255, errors=144]
